In [10]:
# ============================================================
# HYROX BILBAO 2026 — MODELO DE DATOS
# Input:  hyrox_bilbao_2026_master.csv
# Output: hyrox_bilbao_2026.duckdb
# ============================================================

from google.colab import drive
import pandas as pd
import duckdb
import os

drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/hyrox'

print("✅ Drive montado")
print(f"\nArchivos disponibles:")
for a in sorted(os.listdir(RUTA)):
    print(f"  {a}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado

Archivos disponibles:
  hyox_page0.html
  hyrox_bilbao_2026.duckdb
  hyrox_bilbao_2026_clean.csv
  hyrox_bilbao_2026_master.csv
  hyrox_bilbao_2026_raw.csv
  hyrox_bilbao_2026_splits.csv
  hyrox_page1.html
  hyrox_page2.html
  hyrox_page3.html
  hyrox_page4.html
  hyrox_page5.html
  hyrox_page6.html
  hyrox_page7.html
  hyrox_page8.html


In [11]:
# CELDA 2 — Cargar master y crear conexión DuckDB
df_master = pd.read_csv(f'{RUTA}/hyrox_bilbao_2026_master.csv')
print(f"Master cargado: {len(df_master)} registros | {df_master.shape[1]} columnas")

# Crear base de datos DuckDB
con = duckdb.connect(f'{RUTA}/hyrox_bilbao_2026.duckdb')
print("✅ Conexión DuckDB establecida")

Master cargado: 855 registros | 31 columnas
✅ Conexión DuckDB establecida


In [12]:
# CELDA 3 — Crear dim_evento
dim_evento = df_master[['evento', 'año', 'modalidad']].drop_duplicates().reset_index(drop=True)
dim_evento.insert(0, 'id_evento', range(1, len(dim_evento) + 1))

con.execute("DROP TABLE IF EXISTS dim_evento")
con.execute("CREATE TABLE dim_evento AS SELECT * FROM dim_evento")

print("✅ dim_evento creada")
print(dim_evento)

✅ dim_evento creada
   id_evento  evento   año    modalidad
0          1  Bilbao  2026  Doubles Men


In [13]:
# CELDA 4 — Crear dim_categoria
dim_categoria = df_master[['grupo_edad']].drop_duplicates().reset_index(drop=True)
dim_categoria.insert(0, 'id_categoria', range(1, len(dim_categoria) + 1))
dim_categoria = dim_categoria.sort_values('grupo_edad').reset_index(drop=True)
dim_categoria['id_categoria'] = range(1, len(dim_categoria) + 1)

con.execute("DROP TABLE IF EXISTS dim_categoria")
con.execute("CREATE TABLE dim_categoria AS SELECT * FROM dim_categoria")

print("✅ dim_categoria creada")
print(dim_categoria)

✅ dim_categoria creada
   id_categoria grupo_edad
0             1      16-24
1             2      25-29
2             3      30-34
3             4      35-39
4             5      40-44
5             6      45-49
6             7      50-54
7             8      55-59
8             9      60-64


In [14]:
# CELDA 5 — Crear dim_atleta
dim_atleta = df_master[['nombres_pareja', 'atleta_1', 'atleta_2']].drop_duplicates().reset_index(drop=True)
dim_atleta.insert(0, 'id_atleta', range(1, len(dim_atleta) + 1))

con.execute("DROP TABLE IF EXISTS dim_atleta")
con.execute("CREATE TABLE dim_atleta AS SELECT * FROM dim_atleta")

print("✅ dim_atleta creada")
print(f"   {len(dim_atleta)} parejas únicas")
print(dim_atleta.head(5).to_string())

✅ dim_atleta creada
   855 parejas únicas
   id_atleta                                  nombres_pareja                     atleta_1                atleta_2
0          1  Jose Agustin Alises Gimenez, Luis Garcia Rubio  Jose Agustin Alises Gimenez       Luis Garcia Rubio
1          2    Leonardo Alonso Mora, Sergio López Izquierdo         Leonardo Alonso Mora  Sergio López Izquierdo
2          3                Bernardo Branco, Ricardo Fonseca              Bernardo Branco         Ricardo Fonseca
3          4               Raul Sevillano, Pedro Jose Olmedo               Raul Sevillano       Pedro Jose Olmedo
4          5                  Bradley Johnson, Tom Cansfield              Bradley Johnson           Tom Cansfield


In [15]:
# CELDA 6 — Crear fact_resultado
# Unir con dimensiones para obtener los IDs
df_fact = df_master.merge(dim_evento[['id_evento', 'evento', 'año', 'modalidad']], on=['evento', 'año', 'modalidad'])
df_fact = df_fact.merge(dim_categoria[['id_categoria', 'grupo_edad']], on='grupo_edad')
df_fact = df_fact.merge(dim_atleta[['id_atleta', 'nombres_pareja']], on='nombres_pareja')

# Seleccionar columnas para fact
columnas_fact = [
    'id_atleta', 'id_evento', 'id_categoria',
    'pos_general', 'pos_categoria',
    'tiempo_total', 'tiempo_segundos', 'tiempo_minutos',
    'running_1', '1000m_skierg', 'running_2', '50m_sled_push',
    'running_3', '50m_sled_pull', 'running_4', '80m_burpee_broad_jump',
    'running_5', '1000m_row', 'running_6', '200m_farmers_carry',
    'running_7', '100m_sandbag_lunges', 'running_8', 'wall_balls',
    'roxzone_time', 'run_total', 'best_run_lap'
]

df_fact = df_fact[columnas_fact].reset_index(drop=True)
df_fact.insert(0, 'id_resultado', range(1, len(df_fact) + 1))

con.execute("DROP TABLE IF EXISTS fact_resultado")
con.execute("CREATE TABLE fact_resultado AS SELECT * FROM df_fact")

print("✅ fact_resultado creada")
print(f"   {len(df_fact)} registros | {df_fact.shape[1]} columnas")
print(df_fact.head(3).to_string())

✅ fact_resultado creada
   855 registros | 28 columnas
   id_resultado  id_atleta  id_evento  id_categoria  pos_general  pos_categoria tiempo_total  tiempo_segundos  tiempo_minutos running_1 1000m_skierg running_2 50m_sled_push running_3 50m_sled_pull running_4 80m_burpee_broad_jump running_5 1000m_row running_6 200m_farmers_carry running_7 100m_sandbag_lunges running_8 wall_balls roxzone_time run_total best_run_lap
0             1          1          1             2            1              1     00:50:14             3014           50.23  00:03:23     00:03:39  00:03:11      00:01:44  00:03:15      00:02:40  00:03:23              00:01:42  00:03:25  00:03:53  00:03:20           00:01:16  00:03:23            00:02:09  00:03:25   00:03:19     00:03:15  00:26:42     00:03:11
1             2          2          1             2            2              2     00:51:46             3106           51.77  00:03:38     00:03:35  00:03:25      00:01:36  00:03:31      00:02:29  00:03:29         

In [16]:
# CELDA 7 — Verificar modelo con queries analíticas
print("=== TABLAS EN LA BASE DE DATOS ===")
print(con.execute("SHOW TABLES").fetchdf())

print("\n=== QUERY 1: Top 10 parejas por tiempo total ===")
q1 = con.execute("""
    SELECT a.nombres_pareja, a.atleta_1, a.atleta_2,
           c.grupo_edad, f.pos_general, f.tiempo_total
    FROM fact_resultado f
    JOIN dim_atleta a ON f.id_atleta = a.id_atleta
    JOIN dim_categoria c ON f.id_categoria = c.id_categoria
    ORDER BY f.tiempo_segundos
    LIMIT 10
""").fetchdf()
print(q1.to_string())

print("\n=== QUERY 2: Tiempo medio por estación ===")
q2 = con.execute("""
    SELECT
        AVG(tiempo_segundos) / 60 AS media_total_min,
        AVG(CAST(SPLIT_PART("1000m_skierg", ':', 1) AS INT) * 3600 +
            CAST(SPLIT_PART("1000m_skierg", ':', 2) AS INT) * 60 +
            CAST(SPLIT_PART("1000m_skierg", ':', 3) AS INT)) / 60 AS media_skierg_min,
        AVG(CAST(SPLIT_PART("50m_sled_push", ':', 1) AS INT) * 3600 +
            CAST(SPLIT_PART("50m_sled_push", ':', 2) AS INT) * 60 +
            CAST(SPLIT_PART("50m_sled_push", ':', 3) AS INT)) / 60 AS media_sled_push_min
    FROM fact_resultado
""").fetchdf()
print(q2.round(2).to_string())

print("\n=== QUERY 3: Participantes por grupo de edad ===")
q3 = con.execute("""
    SELECT c.grupo_edad, COUNT(*) as parejas,
           ROUND(AVG(f.tiempo_minutos), 1) as media_minutos
    FROM fact_resultado f
    JOIN dim_categoria c ON f.id_categoria = c.id_categoria
    GROUP BY c.grupo_edad
    ORDER BY c.grupo_edad
""").fetchdf()
print(q3.to_string())

=== TABLAS EN LA BASE DE DATOS ===
             name
0      dim_atleta
1   dim_categoria
2      dim_evento
3  fact_resultado

=== QUERY 1: Top 10 parejas por tiempo total ===
                                   nombres_pareja                     atleta_1                    atleta_2 grupo_edad  pos_general tiempo_total
0  Jose Agustin Alises Gimenez, Luis Garcia Rubio  Jose Agustin Alises Gimenez           Luis Garcia Rubio      25-29            1     00:50:14
1    Leonardo Alonso Mora, Sergio López Izquierdo         Leonardo Alonso Mora      Sergio López Izquierdo      25-29            2     00:51:46
2                Bernardo Branco, Ricardo Fonseca              Bernardo Branco             Ricardo Fonseca      25-29            3     00:53:08
3               Raul Sevillano, Pedro Jose Olmedo               Raul Sevillano           Pedro Jose Olmedo      25-29            4     00:54:02
4                  Bradley Johnson, Tom Cansfield              Bradley Johnson               Tom Cansfiel

In [17]:
# CELDA 7b — Crear vista con splits en segundos (útil para Power BI)
con.execute("""
    CREATE OR REPLACE VIEW v_splits_segundos AS
    SELECT
        id_resultado, id_atleta, id_evento, id_categoria,
        pos_general, pos_categoria, tiempo_segundos, tiempo_minutos,
        -- Estaciones en segundos
        CAST(SPLIT_PART("1000m_skierg", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("1000m_skierg", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("1000m_skierg", ':', 3) AS INT) AS skierg_seg,

        CAST(SPLIT_PART("50m_sled_push", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("50m_sled_push", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("50m_sled_push", ':', 3) AS INT) AS sled_push_seg,

        CAST(SPLIT_PART("50m_sled_pull", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("50m_sled_pull", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("50m_sled_pull", ':', 3) AS INT) AS sled_pull_seg,

        CAST(SPLIT_PART("80m_burpee_broad_jump", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("80m_burpee_broad_jump", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("80m_burpee_broad_jump", ':', 3) AS INT) AS burpee_seg,

        CAST(SPLIT_PART("1000m_row", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("1000m_row", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("1000m_row", ':', 3) AS INT) AS row_seg,

        CAST(SPLIT_PART("200m_farmers_carry", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("200m_farmers_carry", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("200m_farmers_carry", ':', 3) AS INT) AS farmers_carry_seg,

        CAST(SPLIT_PART("100m_sandbag_lunges", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("100m_sandbag_lunges", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("100m_sandbag_lunges", ':', 3) AS INT) AS sandbag_lunges_seg,

        CAST(SPLIT_PART("wall_balls", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("wall_balls", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("wall_balls", ':', 3) AS INT) AS wall_balls_seg,

        -- Carreras en segundos
        CAST(SPLIT_PART("running_1", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("running_1", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("running_1", ':', 3) AS INT) AS running_1_seg,

        CAST(SPLIT_PART("run_total", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("run_total", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("run_total", ':', 3) AS INT) AS run_total_seg,

        CAST(SPLIT_PART("roxzone_time", ':', 1) AS INT) * 3600 +
        CAST(SPLIT_PART("roxzone_time", ':', 2) AS INT) * 60 +
        CAST(SPLIT_PART("roxzone_time", ':', 3) AS INT) AS roxzone_seg

    FROM fact_resultado
""")

print("✅ Vista v_splits_segundos creada")
print(con.execute("SELECT * FROM v_splits_segundos LIMIT 3").fetchdf().to_string())

✅ Vista v_splits_segundos creada
   id_resultado  id_atleta  id_evento  id_categoria  pos_general  pos_categoria  tiempo_segundos  tiempo_minutos  skierg_seg  sled_push_seg  sled_pull_seg  burpee_seg  row_seg  farmers_carry_seg  sandbag_lunges_seg  wall_balls_seg  running_1_seg  run_total_seg  roxzone_seg
0             1          1          1             2            1              1             3014           50.23         219            104            160         102      233                 76                 129             199            203           1602          195
1             2          2          1             2            2              2             3106           51.77         215             96            149         116      227                 82                 149             202            218           1684          189
2             3          3          1             2            3              3             3188           53.13         217            112      

In [18]:
# CELDA 8 — Cerrar conexión
con.close()
print("✅ Base de datos guardada en Drive")
print(f"   Ruta: {RUTA}/hyrox_bilbao_2026.duckdb")

✅ Base de datos guardada en Drive
   Ruta: /content/drive/MyDrive/hyrox/hyrox_bilbao_2026.duckdb
